In [32]:
import pandas as pd
import os

#############
# clean_burrow: takes the last 7 characters of a filename and returns the burrow number
#     - removes file extension if present (.txt, .csv, any case)    
#     - takes the last 3 characters
#     - returns the burrow number padded to 3 digits becuase we treat it as charccters
####
def clean_burrow(val: str) -> str:
    s = str(val)

    # Remove file extension if present (.txt, .csv, any case)
    for ext in (".txt", ".csv"):
        if s.lower().endswith(ext):
            s = s[: -len(ext)]
            break
    # take the last 3 of what is left
    s = s[-3:]

    # deal with less than 3 digits
    if s.startswith("_"):       # e.g., "_31"
        s = s[1:]               # drop the leading underscore → "31"
    elif "_" in s:              # e.g., "5_3"
        s = s.split("_")[-1]    # take part after underscore → "3"

    # Return only if it's digits and padded to 3 places
    return s.zfill(3)

# file_path_RFID = "/Users/bobmauck/devel/Combo_App/Example_Data/RF_06_24_2025_933.TXT"   # get_user_file()# Get user file path
file_path_RFID = "/Users/bobmauck/Dropbox/BIG_Science/MOMs/Testing/Sam_Data/RF_07_26_2025_252.TXT"

filename = os.path.basename(file_path_RFID)

# Read CSV without assuming headers
df_RFID = pd.read_csv(file_path_RFID, delimiter=',', header=None, names=['PIT_ID', 'Rdr', 'PIT_DateTime'], on_bad_lines='warn')

OUT_FMT = "%m/%d/%Y %H:%M:%S"

df_RFID["PIT_DateTime"] = pd.to_datetime(df_RFID["PIT_DateTime"],
    errors="coerce",  # handles both "06/14/2025 16:46:52" and "2025-07-26T14:00:00"
    )
df_RFID["PIT_DateTime"] = df_RFID["PIT_DateTime"].dt.strftime(OUT_FMT)
print ("Before cleaning:")
print(df_RFID.head(10))

# Drop known status rows and non‑numeric reader codes
EXPECTED_LEN = 10  # adjust if needed

if True:
    df_RFID = df_RFID[
        ~df_RFID["PIT_ID"].str.upper().isin({"STARTUP", "RUNNING"})
        & pd.to_numeric(df_RFID["Rdr"], errors="coerce").notna()
        & df_RFID["PIT_ID"].str.len().eq(EXPECTED_LEN)
    ].copy()
else:
# Drop known status rows and non‑numeric reader codes
    df_RFID = df_RFID[
    ~df_RFID["PIT_ID"].str.upper().isin({"STARTUP", "RUNNING"})
    & pd.to_numeric(df_RFID["Rdr"], errors="coerce").notna()
    ].copy()

df_RFID = df_RFID.reset_index(drop=True)

df_temp = df_RFID.copy()
df_temp['RF_File'] = filename
df_temp['Burrow'] = df_temp['RF_File'].astype(str).str[-7:-4]
df_temp["Burrow"] = df_temp["Burrow"].astype(str).apply(clean_burrow)
print ("After cleaning:")
print(df_temp.head(10))



Before cleaning:
       PIT_ID   Rdr         PIT_DateTime
0     STARTUP  B252  07/26/2025 12:10:56
1  0620000555     1  07/26/2025 12:11:04
2     RUNNING  B252  07/26/2025 13:00:00
3     RUNNING  B252  07/26/2025 14:00:00
4     RUNNING  B252  07/26/2025 15:00:00
5     RUNNING  B252  07/26/2025 16:00:00
6     RUNNING  B252  07/26/2025 17:00:00
7     RUNNING  B252  07/26/2025 18:00:00
8     RUNNING  B252  07/26/2025 19:00:00
9     RUNNING  B252  07/26/2025 20:00:00
After cleaning:
       PIT_ID Rdr         PIT_DateTime                RF_File Burrow
0  0620000555   1  07/26/2025 12:11:04  RF_07_26_2025_252.TXT    252
1  0620000CDD   1  07/27/2025 00:58:40  RF_07_26_2025_252.TXT    252
2  0620000CDD   2  07/27/2025 00:58:41  RF_07_26_2025_252.TXT    252
3  0620000CDD   2  07/27/2025 00:58:43  RF_07_26_2025_252.TXT    252
4  0620000CDD   2  07/27/2025 00:58:45  RF_07_26_2025_252.TXT    252
5  0620000CDD   2  07/27/2025 00:58:46  RF_07_26_2025_252.TXT    252
6  0620000CDD   2  07/27/2025 00:

In [16]:
### get a file that has been joined
### find all the unique PIT IDs
### order them by burrow and datetime
### step thru and make delta times coming and going, etc.

import pandas as pd
import os


# file_path_Joined = "/Users/bobmauck/devel/Combo_App/Example_Data/RF_06_24_2025_933.TXT"   # get_user_file()# Get user file path
file_path_Joined = "/Users/bobmauck/Dropbox/BIG_Science/MOMs/Testing/Sam_Data/aa_Join_97_STARRED.TXT"

filename = os.path.basename(file_path_Joined)

df_Joined = pd.read_csv(file_path_Joined, sep="	", header=0, usecols=["Burrow", "MOM_Time", "Wt", "RFID", "MOM_File"])

df_Joined = df_Joined.sort_values(["Burrow", "RFID", "MOM_Time"], kind="mergesort").reset_index(drop=True)

df_Joined.head(30)

# Detect year flips in MOM_Time between > 2024 and < 2024
# Works for UNIX timestamps (seconds or milliseconds) and falls back to normal datetime strings.
mom_time_raw = df_Joined["MOM_Time"]
mom_time_num = pd.to_numeric(mom_time_raw, errors="coerce")

if mom_time_num.notna().mean() > 0.80:
    median_abs = mom_time_num.dropna().abs().median()
    unix_unit = "ms" if median_abs >= 1e12 else "s"
    mom_dt = pd.to_datetime(mom_time_num, unit=unix_unit, origin="unix", errors="coerce")
    print(f"Parsed MOM_Time as UNIX timestamps in {unix_unit}.")
else:
    mom_dt = pd.to_datetime(mom_time_raw, errors="coerce")
    print("Parsed MOM_Time as datetime strings.")

year_state = pd.Series(pd.NA, index=df_Joined.index, dtype="object")
year_state[mom_dt.dt.year > 2024] = "gt2024"
year_state[mom_dt.dt.year < 2024] = "lt2024"

state_filtered = year_state.dropna()
prev_state = state_filtered.shift(1)

n_gt_to_lt = int(((prev_state == "gt2024") & (state_filtered == "lt2024")).sum())
n_lt_to_gt = int(((prev_state == "lt2024") & (state_filtered == "gt2024")).sum())
n_total_flips = n_gt_to_lt + n_lt_to_gt

print(f">2024 -> <2024 transitions: {n_gt_to_lt}")
print(f"<2024 -> >2024 transitions: {n_lt_to_gt}")
print(f"Total >/< 2024 transitions: {n_total_flips}")


,Burrow,MOM_File,MOM_Time,Wt,RFID
0,97,DL_06_13_2025_97.TXT,06/13/2025 23:13:40,48.350,0620000AF4
1,97,DL_06_16_2025_97.TXT,06/16/2025 22:24:51,50.230,0620000AF4
2,97,DL_06_20_2025_97.TXT,06/21/2025 03:03:46,43.230,0620000AF4
3,97,DL_06_24_2025_97.TXT,06/25/2025 00:12:12,52.740,0620000AF4
4,97,DL_06_25_2025_97.TXT,06/26/2025 02:55:26,49.010,0620000AF4
5,97,DL_06_25_2025_97.TXT,06/26/2025 02:57:23,47.650,0620000AF4
6,97,DL_06_27_2025_97.TXT,06/27/2025 22:42:53,44.560,0620000AF4
7,97,DL_07_01_2025_97.TXT,07/02/2025 01:19:21,45.600,0620000AF4
8,97,DL_07_04_2025_97.TXT,07/05/2025 03:11:09,40.160,0620000AF4
9,97,DL_08_10_2025_97.TXT,08/11/2025 00:45:08,53.140,0620000AF4


In [2]:
import pandas as pd

path = "/Users/bobmauck/Library/CloudStorage/GoogleDrive-mauckr@kenyon.edu/.shortcut-targets-by-id/1paNSXGkj41CwPOn-VE1BFRt7k3oTzm51/PETREL NSF GRANT/2025 DATA and ANALYSIS/Burrow Scale Daily Files 2025/2025_Weight_Analysis/Burrow_221/Burrow_221_Auto_Batch/DL_07_30_2025_221.txt"

# File format: "<weight>, <unix_timestamp>"
df = pd.read_csv(path, header=None, names=["weight", "unix_ts"], sep=",", engine="c")

# Parse UNIX time (auto-detect seconds vs milliseconds)
df["unix_ts"] = pd.to_numeric(df["unix_ts"], errors="coerce")
median_abs = df["unix_ts"].dropna().abs().median()
unix_unit = "ms" if median_abs >= 1e12 else "s"
dt = pd.to_datetime(df["unix_ts"], unit=unix_unit, origin="unix", errors="coerce")

# Classify each row by year side of 2024
state = pd.Series(pd.NA, index=df.index, dtype="object")
state[dt.dt.year > 2024] = "gt2024"
state[dt.dt.year < 2024] = "lt2024"

# Count transitions in row order
state_filtered = state.dropna()
prev_state = state_filtered.shift(1)

n_gt_to_lt = int(((prev_state == "gt2024") & (state_filtered == "lt2024")).sum())
n_lt_to_gt = int(((prev_state == "lt2024") & (state_filtered == "gt2024")).sum())
n_total = n_gt_to_lt + n_lt_to_gt

print(f"Rows: {len(df)}")
print(f"Parsed datetime rows: {dt.notna().sum()}")
print(f"UNIX unit detected: {unix_unit}")
print(f">2024 -> <2024 transitions: {n_gt_to_lt}")
print(f"<2024 -> >2024 transitions: {n_lt_to_gt}")
print(f"Total transitions: {n_total}")
print(f"Min datetime: {dt.min()}")
print(f"Max datetime: {dt.max()}")



Rows: 4974000
Parsed datetime rows: 4974000
UNIX unit detected: s
>2024 -> <2024 transitions: 1
<2024 -> >2024 transitions: 0
Total transitions: 1
Min datetime: 2000-06-13 00:00:00
Max datetime: 2025-07-30 17:43:23


In [4]:
import pandas as pd

path = "/Users/bobmauck/Library/CloudStorage/GoogleDrive-mauckr@kenyon.edu/.shortcut-targets-by-id/1paNSXGkj41CwPOn-VE1BFRt7k3oTzm51/PETREL NSF GRANT/2025 DATA and ANALYSIS/Burrow Scale Daily Files 2025/2025_Weight_Analysis/Burrow_221/Burrow_221_Auto_Batch/DL_07_30_2025_221.txt"

# Read timestamp column (2nd column) from file: "<weight>, <unix_timestamp>"
ts = pd.read_csv(path, header=None, usecols=[1], names=["ts"], sep=",", engine="c")["ts"]
ts = pd.to_numeric(ts, errors="coerce")

# Detect unix unit automatically
median_abs = ts.dropna().abs().median()
unix_unit = "ms" if median_abs >= 1e12 else "s"

# Convert all timestamps to formatted datetime strings
dt_str = pd.to_datetime(ts, unit=unix_unit, origin="unix", errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

# Count requested timestamps
count_2000_06_13 = int((dt_str == "2000-06-13 00:00:00").sum())
count_2007_09_07 = int((dt_str == "2007-09-07 00:02:00").sum())

print(f"UNIX unit detected: {unix_unit}")
print("Requested timestamp counts:")
print(f"2000-06-13 00:00:00: {count_2000_06_13}")
print(f"2007-09-07 00:02:00: {count_2007_09_07}")

# Find only rows where timestamp changes from previous row
prev_ts = ts.shift(1)
changed = ts.ne(prev_ts) & prev_ts.notna()

out = pd.DataFrame({
    "line_number": ts.index + 1,   # file line number (1-based)
    "prev_ts": prev_ts,
    "new_ts": ts
})[changed].copy()

# Keep only changes > 10 seconds apart
if unix_unit == "ms":
    out["delta_seconds"] = (out["new_ts"] - out["prev_ts"]).abs() / 1000.0
else:
    out["delta_seconds"] = (out["new_ts"] - out["prev_ts"]).abs()

out = out[out["delta_seconds"] > 10].copy()

# Format timestamps
out["before"] = pd.to_datetime(out["prev_ts"], unit=unix_unit, origin="unix", errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
out["after"]  = pd.to_datetime(out["new_ts"],  unit=unix_unit, origin="unix", errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

# Final display
result = out[["line_number", "before", "after", "delta_seconds"]].reset_index(drop=True)
print(f"\nCount of changes > 10 seconds: {len(result)}")
print(result.to_string(index=False))


UNIX unit detected: s
Requested timestamp counts:
2000-06-13 00:00:00: 2400
2007-09-07 00:02:00: 3713700

Count of changes > 10 seconds: 17
 line_number              before               after  delta_seconds
     1257901 2025-07-30 17:43:23 2007-09-07 00:02:00    564774083.0
     1551901 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     1552201 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     1876501 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     1876801 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     2912401 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     2912701 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3316201 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     3316501 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3366001 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     3366301 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3623401 2007-09-07 00:02:00 2000-06-13 

In [5]:
import pandas as pd

path = "/Users/bobmauck/Library/CloudStorage/GoogleDrive-mauckr@kenyon.edu/.shortcut-targets-by-id/1paNSXGkj41CwPOn-VE1BFRt7k3oTzm51/PETREL NSF GRANT/2025 DATA and ANALYSIS/Burrow Scale Daily Files 2025/2025_Weight_Analysis/Burrow_221/Burrow_221_Auto_Batch/DL_07_30_2025_221.txt"

# File format: "<weight>, <unix_timestamp>"
df = pd.read_csv(path, header=None, names=["weight", "ts_raw"], sep=",", engine="c")
df["ts_raw"] = pd.to_numeric(df["ts_raw"], errors="coerce")

# Detect unix unit automatically
median_abs = df["ts_raw"].dropna().abs().median()
unix_unit = "ms" if median_abs >= 1e12 else "s"

# Parse datetimes
df["line_number"] = df.index + 1
df["dt"] = pd.to_datetime(df["ts_raw"], unit=unix_unit, origin="unix", errors="coerce")
df["dt_str"] = df["dt"].dt.strftime("%Y-%m-%d %H:%M:%S")

print(f"UNIX unit detected: {unix_unit}")

# 1) Count requested timestamps
count_2000_06_13 = int((df["dt_str"] == "2000-06-13 00:00:00").sum())
count_2007_09_07 = int((df["dt_str"] == "2007-09-07 00:02:00").sum())

print("\nRequested timestamp counts:")
print(f"2000-06-13 00:00:00: {count_2000_06_13}")
print(f"2007-09-07 00:02:00: {count_2007_09_07}")

# 2) Changes > 10 seconds between consecutive changed timestamps
prev_ts = df["ts_raw"].shift(1)
changed = df["ts_raw"].ne(prev_ts) & prev_ts.notna()

changes = pd.DataFrame({
    "line_number": df["line_number"],
    "prev_ts": prev_ts,
    "new_ts": df["ts_raw"]
})[changed].copy()

if unix_unit == "ms":
    changes["delta_seconds"] = (changes["new_ts"] - changes["prev_ts"]).abs() / 1000.0
else:
    changes["delta_seconds"] = (changes["new_ts"] - changes["prev_ts"]).abs()

changes = changes[changes["delta_seconds"] > 10].copy()
changes["before"] = pd.to_datetime(changes["prev_ts"], unit=unix_unit, origin="unix", errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")
changes["after"] = pd.to_datetime(changes["new_ts"], unit=unix_unit, origin="unix", errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S")

result_changes = changes[["line_number", "before", "after", "delta_seconds"]].reset_index(drop=True)

print(f"\nCount of changes > 10 seconds: {len(result_changes)}")
print(result_changes.to_string(index=False))

# 3) Table: last 5 rows where year > 2024, and first 5 rows where year < 2010
last5_gt_2024 = (
    df[df["dt"].dt.year > 2024][["line_number", "dt_str", "ts_raw"]]
    .tail(5)
    .reset_index(drop=True)
)

first5_lt_2010 = (
    df[df["dt"].dt.year < 2010][["line_number", "dt_str", "ts_raw"]]
    .head(5)
    .reset_index(drop=True)
)

print("\nLast 5 rows with year > 2024:")
print(last5_gt_2024.to_string(index=False))

print("\nFirst 5 rows with year < 2010:")
print(first5_lt_2010.to_string(index=False))


UNIX unit detected: s

Requested timestamp counts:
2000-06-13 00:00:00: 2400
2007-09-07 00:02:00: 3713700

Count of changes > 10 seconds: 17
 line_number              before               after  delta_seconds
     1257901 2025-07-30 17:43:23 2007-09-07 00:02:00    564774083.0
     1551901 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     1552201 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     1876501 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     1876801 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     2912401 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     2912701 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3316201 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     3316501 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3366001 2007-09-07 00:02:00 2000-06-13 00:00:00    228268920.0
     3366301 2000-06-13 00:00:00 2007-09-07 00:02:00    228268920.0
     3623401 2007-09-07 00:02:00 2000-06-13

In [1]:
from pathlib import Path
from datetime import datetime, timezone

def process_and_fix_file(file_path: str, group_size: int = 300, step_seconds: int = 6):
    src = Path(file_path)
    out = Path.cwd() / f"{src.stem}_fixed{src.suffix}"

    def parse_ts(ts_text: str):
        try:
            ts = int(ts_text.strip())
        except ValueError:
            return None, None
        unit = "ms" if abs(ts) >= 10**12 else "s"
        try:
            dt = datetime.fromtimestamp(ts / 1000 if unit == "ms" else ts, tz=timezone.utc)
        except Exception:
            return ts, None
        return ts, dt

    # Pass 1: stats
    rows = 0
    parsed_rows = 0
    rows_gt_2024 = 0
    rows_lt_2024 = 0

    with src.open("r", encoding="utf-8", errors="replace") as fin:
        for line in fin:
            rows += 1
            if "," not in line:
                continue
            _, right = line.split(",", 1)
            ts, dt = parse_ts(right)
            if dt is None:
                continue
            parsed_rows += 1
            if dt.year > 2024:
                rows_gt_2024 += 1
            if dt.year < 2024:
                rows_lt_2024 += 1

    print(f"Rows: {rows}")
    print(f"Parsed datetime rows: {parsed_rows}")
    print(f"# rows >2024: {rows_gt_2024}")
    print(f"# rows <2024: {rows_lt_2024}")

    # Pass 2: fix only if at least one valid (>2024) row exists
    if rows_gt_2024 <= 0:
        print("No rows >2024 found; file not fixed.")
        return None

    cached_valid_ts = None
    invalid_run_count = 0
    current_group_replacement = None

    with src.open("r", encoding="utf-8", errors="replace") as fin, out.open("w", encoding="utf-8") as fout:
        for line in fin:
            if "," not in line:
                fout.write(line)
                continue

            left, right = line.rstrip("\n").split(",", 1)
            ts, dt = parse_ts(right)

            if dt is not None and dt.year > 2024:
                cached_valid_ts = ts
                invalid_run_count = 0
                current_group_replacement = None
                fout.write(line)
                continue

            if cached_valid_ts is not None:
                if invalid_run_count % group_size == 0:
                    # Keep same unit as source
                    step = step_seconds * (1000 if abs(cached_valid_ts) >= 10**12 else 1)
                    cached_valid_ts += step
                    current_group_replacement = cached_valid_ts
                invalid_run_count += 1
                fout.write(f"{left}, {current_group_replacement}\n")
            else:
                fout.write(line)

    print(f"Fixed file saved: {out}")
    return str(out)


In [2]:
myPath = "/Users/bobmauck/Library/CloudStorage/GoogleDrive-mauckr@kenyon.edu/.shortcut-targets-by-id/1paNSXGkj41CwPOn-VE1BFRt7k3oTzm51/PETREL NSF GRANT/2025 DATA and ANALYSIS/Burrow Scale Daily Files 2025/2025_Weight_Analysis/Burrow_221/Burrow_221_Auto_Batch/DL_07_30_2025_221.txt"


process_and_fix_file(myPath)


Rows: 4974000
Parsed datetime rows: 4974000
# rows >2024: 1257900
# rows <2024: 3716100
Fixed file saved: /Users/bobmauck/devel/Combo_MOM_RFID_App/DL_07_30_2025_221_fixed.txt


'/Users/bobmauck/devel/Combo_MOM_RFID_App/DL_07_30_2025_221_fixed.txt'